# AgentCore 메모리 브랜칭을 활용한 Strands 멀티 에이전트 시스템

## 소개

이 노트북은 AWS AgentCore 메모리와 Strands 프레임워크를 사용하여 **메모리 브랜칭이 포함된 멀티 에이전트 시스템**을 구현하는 방법을 보여줍니다. 이 예제는 고급 메모리 기능인 **대화 브랜칭**을 보여줍니다. 이를 통해 에이전트가 원래 대화 스레드를 보존하면서 대화 기록을 포크하고 대안적 대화 경로를 탐색할 수 있습니다.

## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 메모리 브랜칭을 활용한 단기 대화                                                 |
| 에이전트 유스케이스 | 여행 계획 어시스턴트                                                             |
| 에이전트 프레임워크 | Strands 에이전트                                                                 |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, Strands 에이전트, 도구를 통한 메모리 검색                 |
| 난이도              | 입문                                                                             |


학습 내용:

- 대화 기록을 포크하기 위한 메모리 브랜칭 구현 방법
- 다른 대화 브랜치에서 작동하는 전문 에이전트 생성
- 브랜치 수명 주기 관리 (생성, 초기화 및 브랜치 간 전환)
- main 및 브랜치된 대화 모두에서 대화 컨텍스트 유지

### 시나리오 맥락

이 예제에서는 메모리 브랜칭을 보여주는 **여행 계획 시스템**을 생성합니다:
1. 모든 대화가 저장되는 **main 브랜치** 대화
2. 항공편 옵션을 탐색하기 위해 main 대화에서 포크되는 **항공편 에이전트 브랜치**
3. 호텔 옵션을 탐색하기 위해 main 대화에서 포크되는 **호텔 에이전트 브랜치**
4. 두 에이전트 모두 main 브랜치의 공유 대화 기록에 접근 가능

이 접근 방식은 메모리 브랜칭이 다음을 가능하게 하는 방법을 보여줍니다:
- **병렬 탐색**: 에이전트가 main 대화에 영향을 주지 않고 "what-if" 시나리오를 탐색 가능
- **컨텍스트 보존**: 브랜치된 대화가 원래 대화 기록에 대한 접근을 유지
- **전문 워크플로**: 각 브랜치가 원래 컨텍스트에 기반하면서 자체 대화 경로를 따를 수 있음

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항
- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근

환경을 설정하고 공유 메모리 리소스를 생성하는 것부터 시작하겠습니다!

## 1단계: 환경 설정
이 노트북을 실행하는 데 필요한 모든 라이브러리를 가져오고 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

Amazon Bedrock 모델 및 AgentCore에 대한 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os
region = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## 2단계: 공유 메모리 생성
이 섹션에서는 전문 에이전트 간에 공유될 메모리 리소스를 생성합니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")
memory_id = None


In [ ]:
from botocore.exceptions import ClientError

try:
    print("메모리 생성 중...")
    memory_name = memory_name

    # 메모리 리소스 생성
    memory = client.create_memory_and_wait(
        name=memory_name,                       # 이 메모리 저장소의 고유 이름
        description="Travel Agent STM",         # 사람이 읽을 수 있는 설명
        strategies=[],                          # 단기 메모리를 위한 특별한 전략 없음
        event_expiry_days=7,                    # 7일 후 메모리 만료
        max_wait=300,                           # 메모리 생성 대기 최대 시간 (5분)
        poll_interval=10                        # 10초마다 상태 확인
    )

    # 메모리 ID 추출 및 출력
    memory_id = memory['id']
    print(f"메모리가 성공적으로 생성되었습니다. ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하면 해당 ID를 가져옴
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    print(f"오류: {e}")
    import traceback
    traceback.print_exc()

    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"메모리 정리 실패: {cleanup_error}")

### 브랜칭 지원이 포함된 메모리 이해

우리가 생성한 메모리 리소스는 **대화 브랜칭**을 지원하며, 다음을 가능하게 합니다:

1. **공유 세션, 다중 브랜치**: 모든 에이전트가 동일한 `memory_id`, `actor_id`, `session_id`를 사용하되, 다른 `branch_name` 값을 사용
2. **컨텍스트 상속**: 브랜치가 포크될 때, 부모 브랜치(보통 "main")의 대화 기록을 상속
3. **격리된 진화**: 포크 후 각 브랜치는 독립적인 대화 스레드를 유지
4. **브랜치별 검색**: 에이전트가 자신의 특정 브랜치에서 메모리를 검색 가능

이 브랜칭 접근 방식은 전문 에이전트가 공유 세션 기록에 기반하면서 자체 대화 컨텍스트를 유지할 수 있게 합니다.

## 3단계: 브랜칭 지원이 포함된 메모리 훅 프로바이더 생성

이 단계에서는 **메모리 브랜칭**을 구현하는 커스텀 `ShortTermMemoryHook` 클래스를 정의합니다. 이 고급 훅 프로바이더는 기본 메모리 작업을 브랜치 관리 기능으로 확장합니다:

### 주요 기능:
1. **브랜치 관리**: 대화 브랜치를 자동으로 생성하고 초기화
2. **메모리 검색**: 지정된 브랜치에서 대화 기록을 가져옴
3. **메모리 저장**: 적절한 브랜치에 새 대화를 저장
4. **브랜치 포크**: main 대화 스레드에서 새 브랜치를 생성

### 브랜칭 작동 방식:
- 각 에이전트가 `branch_name`을 지정 가능 (기본값은 "main")
- main이 아닌 브랜치는 main 대화에서 자동으로 포크됨
- 브랜치는 포크 시점까지의 대화 기록을 상속
- 포크 후 각 브랜치는 독립적인 대화 흐름을 유지



In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory import MemorySessionManager
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_id: str, region_name: str = "us-west-2", branch_name: str = "main"):
        """Initialize the hook with a MemorySessionManager.

        Args:
            memory_id: The AgentCore Memory ID
            region_name: AWS region for the memory service
            branch_name: Branch name for this agent's memory (default: "main")
        """
        self.memory_manager = MemorySessionManager(
            memory_id=memory_id,
            region_name=region_name
        )
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # 액터/세션 조합별 세션 객체 캐시
        self._branch_initialized = False  # 브랜치 생성 여부 추적

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """Get or create a MemorySession for the given actor/session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(
                actor_id=actor_id,
                session_id=session_id
            )
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """Initialize a branch if it doesn't exist and this is not the main branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
        """
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 브랜치가 이미 존재하는지 확인
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # 포크할 main 브랜치의 마지막 이벤트 가져오기
                main_events = memory_session.list_events(branch_name="main")
                if main_events:
                    last_event = main_events[-1]
                    # 초기 메시지와 함께 브랜치 생성
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(f"Starting {self.branch_name} branch", MessageRole.ASSISTANT)
                        ]
                    )
                    logger.info(f"브랜치 생성 완료: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(f"브랜치 {self.branch_name} 초기화 실패: {e}", exc_info=True)

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트 시작 시 최근 대화 기록 로드"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                return

            # 필요한 경우 브랜치 초기화 (main이 아닌 브랜치)
            if self.branch_name !="main":
                self._initialize_branch(actor_id, session_id)

            # 메모리 세션 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 이 브랜치에서 마지막 5개 대화 턴 가져오기
            recent_turns = memory_session.get_last_k_turns(
                k=5,
                branch_name=self.branch_name
            )

            if recent_turns:
                # 컨텍스트를 위한 대화 기록 포맷팅
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.content.get('role', 'unknown').lower()
                        text = message.content.get('content', {}).get('text', '')
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages)
                    logger.info(f"브랜치 '{self.branch_name}'에서 컨텍스트 로드 ({len(context_messages)}개 메시지)")

                    # 에이전트의 시스템 프롬프트에 컨텍스트 추가
                    event.agent.system_prompt += (
                        f"\n\nRecent conversation history (from {self.branch_name}):\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )

                    logger.info(f"브랜치 '{self.branch_name}'에서 {len(recent_turns)}개의 최근 대화 턴 로드 완료")
            else:
                logger.info(f"브랜치 '{self.branch_name}'에 이전 대화 기록이 없습니다")

        except Exception as e:
            logger.error(f"대화 기록 로드 실패: {e}", exc_info=True)

    def on_message_added(self, event: MessageAddedEvent):
        """적절한 브랜치에 대화 턴 저장"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                return

            # 메모리 세션 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 마지막 메시지 가져오기
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("빈 메시지 건너뜀")
                return

            # 역할 문자열을 MessageRole enum으로 매핑
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # 적절한 브랜치에 메시지 저장
            if self.branch_name == "main":
                # main 브랜치 - 일반적으로 턴 추가
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)]
                )
            else:
                # main이 아닌 브랜치 - 기존 브랜치에 추가 필요
                # 브랜치가 존재하지 않으면 초기화
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # 이 브랜치의 최신 이벤트 가져오기
                branch_events = memory_session.list_events(branch_name=self.branch_name)
                if branch_events:
                    # 브랜치 이름을 지정하여 기존 브랜치에 추가 (rootEventId 없이)
                    memory_session.add_turns(
                        messages=[ConversationalMessage(content_text, message_role)],
                        branch={"name": self.branch_name}
                    )
                else:
                    # _initialize_branch가 작동했다면 발생하지 않지만, 처리
                    logger.warning(f"초기화 후 브랜치 {self.branch_name}를 찾을 수 없습니다")
                    self._initialize_branch(actor_id, session_id)

            logger.debug(f"브랜치 '{self.branch_name}'에 메시지 저장 완료: {role_str}")

        except Exception as e:
            logger.error(f"메시지 저장 실패: {e}", exc_info=True)

    def create_branch(self, actor_id: str, session_id: str,
                      root_event_id: str, branch_name: str,
                      messages: list):
        """Create a new conversation branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
            root_event_id: Event ID to branch from
            branch_name: Name for the new branch
            messages: List of ConversationalMessage objects to add to the branch
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.fork_conversation(
            root_event_id=root_event_id,
            branch_name=branch_name,
            messages=messages
        )

    def list_branches(self, actor_id: str, session_id: str):
        """List all branches for a session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            List of branch information
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.list_branches()

    def get_session(self, actor_id: str, session_id: str):
        """Get the memory session object for direct access.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory hooks with the registry.

        Args:
            registry: The HookRegistry to register callbacks with
        """
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 4단계: 메모리 브랜칭을 활용한 멀티 에이전트 아키텍처 생성

이 섹션에서는 **다른 메모리 브랜치**를 사용하는 전문 에이전트를 생성하여 브랜칭 기능을 보여줍니다:

### 브랜칭 전략:
- **Main 브랜치**: 코디네이터의 대화를 저장하고 기본 대화 스레드 역할
- **flight_agent_memory 브랜치**: 항공편 관련 대화를 위한 별도 브랜치
- **hotel_agent_memory 브랜치**: 호텔 관련 대화를 위한 별도 브랜치

각 전문 에이전트는 자체 브랜치에서 작동하며, 처음 사용 시 main 대화에서 자동으로 포크됩니다. 이를 통해:

- 다른 전문 분야에 대한 독립적인 대화 흐름
- 도메인별 컨텍스트의 격리
- main 대화 스레드의 보존

In [ ]:
# 필요한 구성 요소 가져오기
from strands import Agent, tool

In [ ]:
# 각 전문 에이전트에 대해 고유한 액터 ID를 생성하되 세션 ID는 공유
actor_id = f"travel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"travel/{actor_id}/preferences/"

### 브랜치된 메모리를 가진 전문 에이전트 생성

다음으로 시스템 프롬프트를 정의하고 다른 메모리 브랜치를 사용하는 에이전트를 생성합니다. 동일한 `actor_id`와 `session_id`를 사용하되, 격리된 대화 컨텍스트를 만들기 위해 다른 `branch_name` 값을 사용하는 방법에 주목하세요:

In [ ]:
# 호텔 예약 전문가의 시스템 프롬프트
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# 항공편 예약 전문가의 시스템 프롬프트
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

In [ ]:
flight_memory_hooks = None
hotel_memory_hooks = None

### 브랜치별 메모리를 가진 에이전트 도구 구현

이제 전문 에이전트를 도구로 구현합니다. 각 에이전트는 특정 브랜치 이름으로 구성된 자체 메모리 훅을 갖습니다:
- 항공편 어시스턴트는 `flight_agent_memory` 브랜치 사용
- 호텔 어시스턴트는 `hotel_agent_memory` 브랜치 사용

이 에이전트들이 호출되면:
1. 훅이 브랜치가 존재하는지 확인
2. 존재하지 않으면 main 대화에서 새 브랜치를 포크
3. 에이전트의 대화가 전용 브랜치에 저장됨
4. 에이전트는 여전히 main 브랜치의 컨텍스트에 접근 가능

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    global flight_memory_hooks
    try:
        if flight_memory_hooks is None:
            # "flight_agent_memory" 브랜치 이름으로 훅 생성
            flight_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="flight_agent_memory"
            )

        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"

@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    global hotel_memory_hooks
    try:
        if hotel_memory_hooks is None:
            # "hotel_agent_memory" 브랜치 이름으로 훅 생성
            hotel_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="hotel_agent_memory"
            )

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### 코디네이터 에이전트 생성 (Main 브랜치)

코디네이터 에이전트는 **main 브랜치**에서 작동하며 전문 에이전트에 위임합니다. 다음에 주목하세요:

- 항공편이나 호텔 어시스턴트를 호출하면, 해당 에이전트들이 자체 브랜치를 포크
- 각 전문 에이전트의 브랜치는 main 대화의 현재 상태에서 시작

In [ ]:
# 코디네이터 에이전트의 시스템 프롬프트
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
agent_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
            )

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    hooks=[agent_memory_hooks],
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant],
    state={
        "actor_id": actor_id,
        "session_id": session_id
    }
)

#### 멀티 에이전트 시스템 준비 완료!

## 에이전트를 테스트해 보겠습니다.

여행 계획 시나리오로 멀티 에이전트 시스템을 테스트하겠습니다:

In [ ]:
# LA에서 마드리드까지 7월 1일부터 8월 2일까지 여행 예약 요청
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
# 지금은 항공편에만 집중하고 싶습니다. 영국항공 직항편으로 요청
response = travel_agent("I would only like to focus on the flight at the moment. Direct flight with British Airways")

In [ ]:
print("\n=== 메모리 브랜치 조회 ===")

if flight_memory_hooks or hotel_memory_hooks:
    # 브랜치를 나열하기 위해 아무 메모리 세션이나 가져옴 (모두 같은 세션을 가리킴)
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # 세션의 모든 브랜치 나열
        branches = memory_session.list_branches()
        print(f"\n세션에 총 {len(branches)}개의 브랜치가 있습니다:")
        for branch in branches:
            print(f"  - 브랜치: {branch.name}")
            print(f"    └─ 이벤트: {len(memory_session.list_events(branch_name=branch.name))}")
            print(f"    └─ 생성일: {branch.created}")

        print("\n각 브랜치는 다른 에이전트의 메모리를 나타냅니다:")
        print("  • 'main' = 여행 코디네이터 대화")
        print("  • 'flight_agent_memory' = 항공편 어시스턴트 대화")
        print("  • 'hotel_agent_memory' = 호텔 어시스턴트 대화")

In [ ]:
print("\n=== 브랜치별 이벤트 접근 ===")

if flight_memory_hooks or hotel_memory_hooks:
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # main 브랜치 (코디네이터)의 이벤트 가져오기
        main_events = memory_session.list_events(branch_name="main")
        print(f"\nMain 브랜치 - 코디네이터 ({len(main_events)}개 이벤트):")
        if main_events:
            for event in main_events[-3:]:  # 마지막 3개 이벤트 표시
                for payload in event.payload:
                    if 'conversational' in payload:
                        role = payload['conversational']['role']
                        text = payload['conversational']['content']['text']
                        print(f"  {role}: {text[:100]}...")
        else:
            print("  main 브랜치에 이벤트가 없습니다")

        # 항공편 에이전트 브랜치의 이벤트 가져오기
        try:
            flight_branch_events = memory_session.list_events(branch_name="flight_agent_memory")
            print(f"\n항공편 에이전트 브랜치 ({len(flight_branch_events)}개 이벤트):")
            if flight_branch_events:
                print("모든 항공편 관련 대화가 여기에 저장됩니다:")
                for event in flight_branch_events[-3:]:  # 마지막 3개 이벤트 표시
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  이벤트 없음 - 항공편 어시스턴트가 아직 호출되지 않았습니다")
        except Exception as e:
            print(f"  항공편 브랜치가 아직 생성되지 않았습니다: {e}")

        # 호텔 에이전트 브랜치의 이벤트 가져오기
        try:
            hotel_branch_events = memory_session.list_events(branch_name="hotel_agent_memory")
            print(f"\n호텔 에이전트 브랜치 ({len(hotel_branch_events)}개 이벤트):")
            if hotel_branch_events:
                print("모든 호텔 관련 대화가 여기에 저장됩니다:")
                for event in hotel_branch_events[-3:]:  # 마지막 3개 이벤트 표시
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  이벤트 없음 - 호텔 어시스턴트가 아직 호출되지 않았습니다")
        except Exception as e:
            print(f"  호텔 브랜치가 아직 생성되지 않았습니다: {e}")

## 요약

이 노트북에서 다음을 보여주었습니다:

1. **메모리 브랜칭 기본 사항**: AgentCore 메모리에서 대화 브랜치를 생성하고 관리하는 방법
2. **브랜치별 에이전트**: 다른 메모리 브랜치에서 작동하는 전문 에이전트 구현 방법
3. **자동 브랜치 생성**: 처음 사용 시 main 대화에서 브랜치가 자동으로 포크되는 방법
4. **브랜치 지속성**: 각 브랜치가 독립적인 대화 기록을 유지하는 방법
5. **컨텍스트 상속**: 브랜치된 대화가 포크 시점까지 main 브랜치의 컨텍스트를 상속하는 방법

### 메모리 브랜칭의 주요 이점:
- **격리**: 각 에이전트가 다른 에이전트를 간섭하지 않고 자체 대화 컨텍스트를 유지
- **유연성**: main 스레드에 영향을 주지 않고 대안적 대화 경로를 탐색
- **조직화**: 도메인별 대화를 별도 브랜치로 정리
- **지속성**: 브랜치별 메모리가 에이전트 인스턴스 간에 지속

이 메모리 브랜칭 아키텍처는 다른 에이전트가 별도이면서 관련된 대화 컨텍스트를 유지해야 하는 정교한 멀티 에이전트 시스템을 구축하기 위한 강력한 접근 방식을 제공합니다.

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다.

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)